# Stop Words

Part 4 of 4 - previous: `01c-lemmatization.ipynb` - next: bag of words / TF-IDF

**What?** Words that appear in almost every English sentence (`a`, `the`, `is`, `in`, `of`, `to` ...) and carry little meaning on their own. NLTK ships curated stop word lists for 30+ languages.

**Why learn it?** Stop words are ~30-40% of any English text (40% in the speech below) but have almost zero *discriminative* power: every document contains them, so for BOW/TF-IDF they only bloat the vocabulary and dilute the features. Search engines have filtered them out since the 1960s for exactly this reason.

Where this sits in the pipeline: **tokenize -> lowercase -> remove stop words -> stem/lemmatize**.

## Setup

One-time NLTK data download (cached in `~/nltk_data`). We need `stopwords` (the word lists) and `punkt_tab` (for `word_tokenize`).

In [1]:
import nltk

nltk.download("stopwords")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/shubham/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/shubham/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## The stop word list

**What?** `stopwords.words("english")` returns NLTK's English list. We wrap it in a `set` - membership checks (`t not in stop_words`) are much faster on a set than on a list.

**Why look at the list itself?** You should know exactly what you are removing - notice it also contains `not`, `no`, `nor` and question words like `who`, `what`, `why`. That matters below.

In [2]:
from nltk.corpus import stopwords

print("languages available:", len(stopwords.fileids()))

stop_words = set(stopwords.words("english"))
print("english stop words:", len(stop_words))
print("first few (alphabetical):", sorted(stop_words)[:10])

for w in ["not", "no", "who", "what"]:
    print(f"  {w!r:<8} in list: {w in stop_words}")

languages available: 33
english stop words: 198
first few (alphabetical): ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']
  'not'    in list: True
  'no'     in list: True
  'who'    in list: True
  'what'   in list: True


## The example: Dr. A.P.J. Abdul Kalam to students

From *"Succeeding Against Established Patterns"* - his address at the Children's Science Congress, Science City, Ahmedabad (5 January 2005). The famous bumble bee passage - keep reading to the last line.

(Source: *President Dr. APJ Abdul Kalam - Selected Speeches, Vol. II*, Publications Division, Government of India.)

In [3]:
speech = """
I am indeed delighted to interact with the students participating in the Children's Science Congress organised by the Indian Science Congress Association. My greetings to the student community and organizers of this science congress. I am happy that this event is integrating millions of children in the age group of 10 to 17 to interact in a common scientific platform. The topic I have chosen for discussion is 'Innovative Minds'.

According to the laws of aerodynamics the bumble bee should be unable to fly. Because of the size, weight, and shape of its body in relationship to the total wing span, flying is scientifically impossible. The bumble bee, being ignorant of scientific theory, keeps fluttering her wings. This high frequency vibration creates a vortex which enables it to fly. With determined efforts you can always succeed against established patterns.
"""

## Removing stop words

**What?** Tokenize the lowercased text, keep only tokens that are alphanumeric (drops punctuation) and not in the stop set.

**Why lowercase first?** The stop list is all lowercase - `The` would slip past the filter. **Why `isalnum()`?** Punctuation tokens are just as content-free as stop words for bag-of-words purposes.

In [4]:
from nltk.tokenize import word_tokenize

tokens = word_tokenize(speech.lower())
filtered = [t for t in tokens if t.isalnum() and t not in stop_words]

print("tokens before:", len(tokens))
print("tokens after :", len(filtered))
print(f"removed {len(tokens) - len(filtered)} tokens ({(len(tokens) - len(filtered)) / len(tokens):.0%} of the speech)")
print()
print(" ".join(filtered))

tokens before: 156
tokens after : 76
removed 80 tokens (51% of the speech)

indeed delighted interact students participating children science congress organised indian science congress association greetings student community organizers science congress happy event integrating millions children age group 10 17 interact common scientific platform topic chosen discussion innovative minds according laws aerodynamics bumble bee unable fly size weight shape body relationship total wing span flying scientifically impossible bumble bee ignorant scientific theory keeps fluttering wings high frequency vibration creates vortex enables fly determined efforts always succeed established patterns


Look at the output above: it reads like a summary of the speech - `students ... science congress ... bumble bee ... fluttering wings ... succeed against established patterns`. Almost all grammar is gone; the content words survived.

**What actually got removed?** Inspect it instead of trusting the name "stop words":

In [5]:
from collections import Counter

removed = [t for t in tokens if t in stop_words]
print("most common removed:", Counter(removed).most_common(8))

most common removed: [('the', 11), ('to', 8), ('of', 7), ('in', 4), ('i', 3), ('this', 3), ('is', 3), ('am', 2)]


## When NOT to remove stop words

Removal helps BOW/TF-IDF-style features, but hurts whenever the "grammar words" carry meaning:

- **Sentiment**: `not good` -> removing `not` (it IS in NLTK's list) leaves `good` - the meaning flips.
- **Questions / search queries**: `who`, `what`, `where` often are the query.
- **Translation, POS/grammar tasks, LLM prompts**: modern transformers use stop words as context - feed them full text.

Rule of thumb: remove stop words for classic count-based features; keep them for sequence models and anything meaning-sensitive.

## Takeaways

- Stop words = high frequency, low information; removing them cut this speech roughly in half while keeping the meaning readable.
- Lowercase before filtering; store the list as a `set`.
- Know your list: NLTK's includes negations (`not`, `no`, `nor`) - fine for topic features, dangerous for sentiment.
- scikit-learn's `CountVectorizer(stop_words="english")` uses its **own** list (not NLTK's) - you will meet it in the BOW/TF-IDF notebooks.